In [74]:
# !pip install "git+https://github.com/VikParuchuri/marker.git" -q

In [75]:
# !pip list | findstr marker

In [76]:
# import sys
# !{sys.executable} -m pip install langdetect

In [77]:
# !pip install ipywidgets

In [78]:
# !pip install -U langchain langchain-core langchain-community

In [79]:
# !pip install langchain-core

In [80]:
import os
import json
from pathlib import Path
from datetime import datetime
from langdetect import detect, LangDetectException
import torch
from tqdm import tqdm

In [81]:
# Создаём папки
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")
FIGURES_DIR = OUTPUT_DIR / "figures"
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"📁 Входная папка: {INPUT_DIR.resolve()}")
print(f"📤 Выходная папка: {OUTPUT_DIR.resolve()}")

📁 Входная папка: C:\Users\user\rag_ref_bot\input
📤 Выходная папка: C:\Users\user\rag_ref_bot\output


In [82]:
# Проверяем наличие PDF
pdf_files = list(INPUT_DIR.glob("*.pdf"))
if not pdf_files:
    print("⚠️ Нет PDF-файлов в папке 'input'. Загрузите файлы и перезапустите эту ячейку.")
else:
    print(f"📄 Найдено PDF-файлов: {len(pdf_files)}")

📄 Найдено PDF-файлов: 1


In [83]:
# Загрузка моделей Marker (использует GPU автоматически)
print("⏳ Загрузка моделей Marker...")

from marker.models import create_model_dict
from marker.converters.pdf import PdfConverter
from marker.output import text_from_rendered

# Загружаем все модели один раз
models = create_model_dict()
print("✅ Модели загружены!")

⏳ Загрузка моделей Marker...
✅ Модели загружены!


In [103]:
import re
import hashlib  # ← добавлено

def extract_title_from_markdown(md_text: str) -> str:
    """Извлекает первый заголовок уровня 1 (# ...) как название."""
    match = re.search(r'^#\s+(.+)$', md_text, re.MULTILINE)
    if match:
        title = match.group(1).strip()
        title = re.sub(r'\*\*(.*?)\*\*', r'\1', title)
        return title
    # Резерв: первая непустая строка
    for line in md_text.split("\n"):
        clean = line.strip("# \t\n")
        if len(clean) > 5 and not clean.startswith(("Figure", "Fig.", "Table", "Рис.", "Таблица")):
            return clean
    return "Unknown"

def safe_filename(text: str, max_len: int = 80) -> str:
    """Создаёт безопасное имя файла, сохраняя кириллицу."""
    if not text or not text.strip():
        return "без_названия"
    safe = re.sub(r'[^\w\s\-\.\u0400-\u04FF]', '_', text)
    return re.sub(r'_+', '_', safe).strip('_')[:max_len]

def make_unique_basename(title: str, original_filename: str) -> str:
    """Генерирует уникальное имя вида: Title_abc123"""
    base = safe_filename(title)
    hash_suffix = hashlib.md5(original_filename.encode()).hexdigest()[:6]  # ← использует hashlib
    return f"{base}_{hash_suffix}"

def detect_language_from_text(text: str) -> str:
    """Определяет язык текста (en/ru/other)."""
    try:
        lang = detect(text[:600])
        return lang if lang in ("en", "ru") else "other"
    except (LangDetectException, IndexError):
        return "unknown"

def fix_image_paths(md_text, base_name):
    prefix = f"figures/{base_name}/"
    
    md_text = re.sub(
        r'!\[\]\((?!http)([^)]+)\)',  
        lambda m: f"![]({prefix}{m.group(1)})",
        md_text
    )
# def make_display_name(pdf_path: str) -> str: 
#     return Path(pdf_path).stem.strip()

In [105]:
all_documents = []

for pdf_path in sorted(pdf_files):
    print(f"\n📄 Обработка: {pdf_path.name}")
    
    try:
        # Конвертация PDF → Markdown + метаданные + изображения
        converter = PdfConverter(artifact_dict=models)
        rendered = converter(str(pdf_path))
        md_text, metadata, images = text_from_rendered(rendered)
        
        # Извлечение названия из H1
        # title = extract_title_from_markdown(md_text)

        # Используем имя файла (без расширения) как название
        title = pdf_path.stem  # .stem возвращает имя файла без расширения

        # Определение языка
        detected_lang = detect_language_from_text(md_text)
        
        # Генерация уникального имени файла
        base_name = make_unique_basename(title, pdf_path.name)

        # display_name = make_display_name(pdf_path)

        md_text = fix_image_paths(md_text, base_name)
        
        # Сохранение Markdown
        md_file = OUTPUT_DIR / f"{base_name}.md"
        md_file.write_text(md_text, encoding="utf-8")
        
        # Сохранение изображений
        saved_figures = []
        if images:
            fig_subdir = FIGURES_DIR / base_name
            fig_subdir.mkdir(exist_ok=True)
            for img_name, img_pil in images.items():
                img_file = fig_subdir / img_name
                img_pil.save(img_file)
                saved_figures.append(str(img_file.relative_to(OUTPUT_DIR)))
        
        # Сбор метаданных
        doc_metadata = {
            "source_pdf": str(pdf_path.resolve()),
            "filename_original": pdf_path.name,          # ✅ оригинальное имя
            # "display_name": display_name,
            "title": title,
            "detected_language": detected_lang,
            "date_processed": datetime.now().isoformat(),
            "markdown_path": str(md_file.relative_to(OUTPUT_DIR)),
            "figures": saved_figures,
            "figure_count": len(saved_figures)
        }
        
        # Сохранение JSON
        json_file = OUTPUT_DIR / f"{base_name}.json"
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(doc_metadata, f, ensure_ascii=False, indent=2)
        
        # Добавление в список документов (для RAG)
        from langchain_core.documents import Document
        doc = Document(page_content=md_text, metadata=doc_metadata)
        all_documents.append(doc)
        
        print(f"✅ Сохранено: {base_name}")
        
    except Exception as e:
        print(f"❌ Ошибка при обработке {pdf_path.name}: {e}")
        continue

print(f"\n🎉 Обработка завершена! Успешно обработано: {len(all_documents)} документов.")


📄 Обработка: Регламент бетонирования при отрицательных температурах.pdf


Recognizing Text: 100%|████████████████████████████████████████████████████████████████| 40/40 [01:12<00:00,  1.81s/it]


❌ Ошибка при обработке Регламент бетонирования при отрицательных температурах.pdf: data must be str, not NoneType

🎉 Обработка завершена! Успешно обработано: 0 документов.


In [101]:
display_name = make_display_name(pdf_path)
display_name

'Регламент бетонирования при отрицательных температурах'

In [86]:
# if all_documents:
#     doc = all_documents[0]
#     print("=== Пример метаданных ===")
#     print(json.dumps({
#         k: v for k, v in doc.metadata.items() 
#         if k in ["title", "filename_original", "detected_language", "figure_count"]
#     }, ensure_ascii=False, indent=2))
    
#     print("\n=== Первые 300 символов Markdown ===")
#     print(doc.page_content[:300] + "...")
# else:
#     print("Нет обработанных документов.")